# Task 1: Client Requests Analysis

In [8]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [9]:
# Load dataset
fact_events = pd.read_csv("datasets/fact_events.csv")
dim_stores = pd.read_csv("datasets/dim_stores.csv")
dim_products = pd.read_csv("datasets/dim_products.csv")
dim_campaigns = pd.read_csv("datasets/dim_campaigns.csv")

print("fact_events:", fact_events.shape)
print("dim_stores:", dim_stores.shape)
print("dim_products:", dim_products.shape)
print("dim_campaigns:", dim_campaigns.shape)

fact_events: (1510, 9)
dim_stores: (50, 2)
dim_products: (15, 3)
dim_campaigns: (2, 4)


## 1) Remove duplicate event rows

In [10]:
events = fact_events.copy()
rows_before = len(events)
print(rows_before, "rows before deduplication")
events = events.drop_duplicates(subset=["store_id", "campaign_id", "product_code"]).copy()
rows_after = len(events)
print(rows_after, "rows after deduplication")
duplicate_rows_removed = rows_before - rows_after

print("Duplicate rows removed:", duplicate_rows_removed)
print("Rows after deduplication:", rows_after)

1510 rows before deduplication
1500 rows after deduplication
Duplicate rows removed: 10
Rows after deduplication: 1500


## 2) Cities with more than 5 stores

In [11]:
store_count_by_city = dim_stores.groupby("city")["store_id"].nunique().sort_values(ascending=False)
num_cities_gt5 = int((store_count_by_city > 5).sum())

print("Number of cities with more than 5 stores:", num_cities_gt5)
store_count_by_city

Number of cities with more than 5 stores: 3


city
Bengaluru        10
Chennai           8
Hyderabad         7
Coimbatore        5
Visakhapatnam     5
Madurai           4
Mysuru            4
Mangalore         3
Trivandrum        2
Vijayawada        2
Name: store_id, dtype: int64

## 3) Fill missing `quantity_sold(before_promo)` with median

In [12]:
missing_before_fill = int(events["quantity_sold(before_promo)"].isna().sum())
print("Missing values before fill:", missing_before_fill)
median_before_qty = float(events["quantity_sold(before_promo)"].median())

events["quantity_sold(before_promo)"] = events["quantity_sold(before_promo)"].fillna(median_before_qty)
missing_after_fill = int(events["quantity_sold(before_promo)"].isna().sum())

print("Missing values filled:", missing_before_fill)
print("Median used for imputation:", median_before_qty)
print("Missing after fill:", missing_after_fill)

Missing values before fill: 20
Missing values filled: 20
Median used for imputation: 78.0
Missing after fill: 0


## 4) Product category with the lowest base price before promotion

In [13]:
min_base_price = events["base_price(before_promo)"].min()
lowest_price_products = events.loc[
    events["base_price(before_promo)"] == min_base_price, ["product_code", "base_price(before_promo)"]
].drop_duplicates()

lowest_price_with_category = lowest_price_products.merge(dim_products, on="product_code", how="left")

print("Lowest base price before promo:", min_base_price)
print("Category(s) at this price:", lowest_price_with_category["category"].dropna().unique().tolist())
lowest_price_with_category

Lowest base price before promo: 50
Category(s) at this price: ['Personal Care']


,product_code,base_price(before_promo),product_name,category
0,P10,50,Atliq_Cream_Beauty_Bathing_Soap (125GM),Personal Care


## 5) Total quantity sold after promo for `BOGOF` during `Diwali`

In [14]:
events_campaign = events.merge(
    dim_campaigns[["campaign_id", "campaign_name"]], on="campaign_id", how="left"
)

total_qty_bogof_diwali = int(
    events_campaign.loc[
        (events_campaign["promo_type"] == "BOGOF")
        & (events_campaign["campaign_name"] == "Diwali"),
        "quantity_sold(after_promo)",
    ].sum()
)

print("Total quantity sold after promo (BOGOF, Diwali):", total_qty_bogof_diwali)

Total quantity sold after promo (BOGOF, Diwali): 34461


## 6) Store with highest quantity sold after promo during `Diwali`

In [15]:
diwali_store_sales = (
    events_campaign.loc[events_campaign["campaign_name"] == "Diwali"]
    .groupby("store_id", as_index=False)["quantity_sold(after_promo)"]
    .sum()
    .sort_values("quantity_sold(after_promo)", ascending=False)
)

top_diwali_store = diwali_store_sales.head(1)
print("Store with highest quantity sold after promo during Diwali:")
top_diwali_store

Store with highest quantity sold after promo during Diwali:


,store_id,quantity_sold(after_promo)
19,STCHE-4,5013


## 7) Compare Sankranti vs Diwali sales lift (before vs after promo quantities)

In [16]:
campaign_qty_compare = (
    events_campaign.loc[events_campaign["campaign_name"].isin(["Sankranti", "Diwali"])]
    .groupby("campaign_name", as_index=False)[
        ["quantity_sold(before_promo)", "quantity_sold(after_promo)"]
    ]
    .sum()
)

campaign_qty_compare["increase_in_qty"] = (
    campaign_qty_compare["quantity_sold(after_promo)"]
    - campaign_qty_compare["quantity_sold(before_promo)"]
)

best_campaign = campaign_qty_compare.sort_values("increase_in_qty", ascending=False).head(1)

campaign_qty_compare

,campaign_name,quantity_sold(before_promo),quantity_sold(after_promo),increase_in_qty
0,Diwali,"109,756.00",183404,"73,648.00"
1,Sankranti,"97,894.00",252069,"154,175.00"


In [17]:
print("Campaign with greater increase in sales:")
best_campaign

Campaign with greater increase in sales:


,campaign_name,quantity_sold(before_promo),quantity_sold(after_promo),increase_in_qty
1,Sankranti,"97,894.00",252069,"154,175.00"


## 8) Product with highest IR% during `Sankranti`

In [18]:
events_campaign["revenue_before"] = (
    events_campaign["base_price(before_promo)"] * events_campaign["quantity_sold(before_promo)"]
)
events_campaign["revenue_after"] = (
    events_campaign["base_price(after_promo)"] * events_campaign["quantity_sold(after_promo)"]
)

sankranti_product_ir = (
    events_campaign.loc[events_campaign["campaign_name"] == "Sankranti"]
    .groupby("product_code", as_index=False)[["revenue_before", "revenue_after"]]
    .sum()
)

sankranti_product_ir["IR%"] = np.where(
    sankranti_product_ir["revenue_before"] == 0,
    np.nan,
    (sankranti_product_ir["revenue_after"] - sankranti_product_ir["revenue_before"])
    / sankranti_product_ir["revenue_before"]
    * 100,
)

sankranti_product_ir = sankranti_product_ir.merge(
    dim_products[["product_code", "product_name"]], on="product_code", how="left"
)

top_product_ir = sankranti_product_ir.sort_values("IR%", ascending=False).head(1)
print("Product with highest IR% during Sankranti:")
top_product_ir[["product_code", "product_name", "IR%"]]

Product with highest IR% during Sankranti:


,product_code,product_name,IR%
2,P03,Atliq_Suflower_Oil (1L),91.83


## 9) Visakhapatnam store with lowest ISU% during `Diwali`

In [19]:
diwali_store_isu = (
    events_campaign.loc[events_campaign["campaign_name"] == "Diwali"]
    .groupby("store_id", as_index=False)[
        ["quantity_sold(before_promo)", "quantity_sold(after_promo)"]
    ]
    .sum()
)

diwali_store_isu = diwali_store_isu.merge(dim_stores, on="store_id", how="left")

# ISU% = ((after - before) / before) * 100
diwali_store_isu["ISU%"] = np.where(
    diwali_store_isu["quantity_sold(before_promo)"] == 0,
    np.nan,
    (diwali_store_isu["quantity_sold(after_promo)"] - diwali_store_isu["quantity_sold(before_promo)"])
    / diwali_store_isu["quantity_sold(before_promo)"]
    * 100,
)

visakhapatnam_lowest_isu = (
    diwali_store_isu.loc[diwali_store_isu["city"] == "Visakhapatnam"]
    .sort_values("ISU%", ascending=True)
    .head(1)
)

print("Visakhapatnam store with lowest ISU% during Diwali:")
visakhapatnam_lowest_isu[["store_id", "city", "ISU%"]]

Visakhapatnam store with lowest ISU% during Diwali:


,store_id,city,ISU%
48,STVSK-3,Visakhapatnam,49.21


## 10) Promo type with both negative IR% and ISU% during `Sankranti`

In [20]:
sankranti_promo_perf = (
    events_campaign.loc[events_campaign["campaign_name"] == "Sankranti"]
    .groupby("promo_type", as_index=False)[
        [
            "revenue_before",
            "revenue_after",
            "quantity_sold(before_promo)",
            "quantity_sold(after_promo)",
        ]
    ]
    .sum()
)

sankranti_promo_perf["IR%"] = np.where(
    sankranti_promo_perf["revenue_before"] == 0,
    np.nan,
    (sankranti_promo_perf["revenue_after"] - sankranti_promo_perf["revenue_before"])
    / sankranti_promo_perf["revenue_before"]
    * 100,
)

sankranti_promo_perf["ISU%"] = np.where(
    sankranti_promo_perf["quantity_sold(before_promo)"] == 0,
    np.nan,
    (sankranti_promo_perf["quantity_sold(after_promo)"] - sankranti_promo_perf["quantity_sold(before_promo)"])
    / sankranti_promo_perf["quantity_sold(before_promo)"]
    * 100,
)

negative_ir_isu = sankranti_promo_perf.loc[
    (sankranti_promo_perf["IR%"] < 0) & (sankranti_promo_perf["ISU%"] < 0),
    ["promo_type", "IR%", "ISU%"],
]

print("Promo type(s) with both negative IR% and ISU% during Sankranti:")
negative_ir_isu

Promo type(s) with both negative IR% and ISU% during Sankranti:


,promo_type,IR%,ISU%
0,25% OFF,-39.33,-19.60


## Final Answers Snapshot

In [23]:
final_answers = {
    "1_duplicate_rows_removed": int(duplicate_rows_removed),
    "2_cities_more_than_5_stores": int(num_cities_gt5),
    "3_missing_values_filled": int(missing_before_fill),
    "3_median_used": float(median_before_qty),
    "4_lowest_base_price": float(min_base_price),
    "4_category_at_lowest_price": lowest_price_with_category["category"].dropna().unique().tolist(),
    "5_total_qty_after_bogof_diwali": int(total_qty_bogof_diwali),
    "6_top_diwali_store": top_diwali_store.to_dict("records"),
    "7_campaign_comparison": campaign_qty_compare.to_dict("records"),
    "8_top_product_ir": top_product_ir[["product_code", "product_name", "IR%"]].round(2).to_dict("records"),
    "9_lowest_isu_visakhapatnam": visakhapatnam_lowest_isu[["store_id", "city", "ISU%"]].round(2).to_dict("records"),
    "10_negative_ir_and_isu_promo_types": negative_ir_isu.round(2).to_dict("records"),
}

final_answers


{'1_duplicate_rows_removed': 10,
 '2_cities_more_than_5_stores': 3,
 '3_missing_values_filled': 20,
 '3_median_used': 78.0,
 '4_lowest_base_price': 50.0,
 '4_category_at_lowest_price': ['Personal Care'],
 '5_total_qty_after_bogof_diwali': 34461,
 '6_top_diwali_store': [{'store_id': 'STCHE-4',
   'quantity_sold(after_promo)': 5013}],
 '7_campaign_comparison': [{'campaign_name': 'Diwali',
   'quantity_sold(before_promo)': 109756.0,
   'quantity_sold(after_promo)': 183404,
   'increase_in_qty': 73648.0},
  {'campaign_name': 'Sankranti',
   'quantity_sold(before_promo)': 97894.0,
   'quantity_sold(after_promo)': 252069,
   'increase_in_qty': 154175.0}],
 '8_top_product_ir': [{'product_code': 'P03',
   'product_name': 'Atliq_Suflower_Oil (1L)',
   'IR%': 91.83}],
 '9_lowest_isu_visakhapatnam': [{'store_id': 'STVSK-3',
   'city': 'Visakhapatnam',
   'ISU%': 49.21}],
 '10_negative_ir_and_isu_promo_types': [{'promo_type': '25% OFF',
   'IR%': -39.33,
   'ISU%': -19.6}]}